<details>
<summary><b>Info</b></summary>

**Last Execution:** 2026-07-25

| Package | Version |
|---------|---------|
| **nnsight** | **0.8** |
| Python | 3.12.3 |
| torch | 2.10.0+cu128 |
| transformers | 5.2.0 |

</details>


# Accessing Intermediate Operations

`.output` and `.input` let you hook into a module's inputs and outputs. But what about the operations *inside* a module's forward pass? `.source` exposes every call site in a module's `forward` as a hookable operation — function calls, method calls, tensor operations — so you can read, replace, or skip a value that lives *between* two operations, with no submodule to attach to.

## Setup

In [1]:
from nnsight.modeling.transformers import TransformersModel

model = TransformersModel("openai-community/gpt2", device_map="auto", dispatch=True)

/home/localjadenfk/miniconda3/envs/ndif2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Discovering Operations

Print `.source` on any module to see its forward method with all hookable operations labeled at their call site (this works outside a trace).

In [2]:
print(model.transformer.h[0].mlp.source)

                    * def forward(self, hidden_states: tuple[torch.FloatTensor] | None) -> torch.FloatTensor:
 self_c_fc_0    ->  0     hidden_states = self.c_fc(hidden_states)
 self_act_0     ->  1     hidden_states = self.act(hidden_states)
 self_c_proj_0  ->  2     hidden_states = self.c_proj(hidden_states)
 self_dropout_0 ->  3     hidden_states = self.dropout(hidden_states)
                    4     return hidden_states
                    5 


Each labeled line (like `self_c_fc_0`, `self_act_0`) is an operation you can access inside a trace. Operation names are the **full dotted callee** joined with `_`, plus a per-name occurrence index in execution order — so `self.c_fc(...)` becomes `self_c_fc_0` and a second call to the same callee would be `..._1`.

Larger modules have more operations. Here's the attention module:

In [3]:
print(model.transformer.h[0].attn.source)

                                             * def forward(
                                             0     self,
                                             1     hidden_states: tuple[torch.FloatTensor] | None,
                                             2     past_key_values: Cache | None = None,
                                             3     attention_mask: torch.FloatTensor | None = None,
                                             4     encoder_hidden_states: torch.Tensor | None = None,
                                             5     encoder_attention_mask: torch.FloatTensor | None = None,
                                             6     output_attentions: bool | None = False,
                                             7     **kwargs,
                                             8 ) -> tuple[torch.Tensor | tuple[torch.Tensor], ...]:
                                             9     is_cross_attention = encoder_hidden_states is not None
                           

## Getting an Intermediate Value

Access any labeled operation's `.output` inside a trace — just like you would with a module. Here we grab the hidden states right after the GELU activation, before the down projection.

In [4]:
with model.trace("The Eiffel Tower is in the city of"):
    post_gelu = model.transformer.h[0].mlp.source.self_act_0.output.save()

print(f"Post-GELU shape: {post_gelu.shape}")

Post-GELU shape: torch.Size([1, 10, 3072])


<details class="admonition note">
<summary>How source works</summary>

On first `.source` access, nnsight rewrites the module's `forward` so each call `fn(*args, **kwargs)` is bracketed by the interleaver. Each operation then exposes `.input` / `.inputs` / `.output` / `.skip` — the same handles a module has, one level finer. The instrumentation is installed lazily and is inert outside a trace, so normal inference is unaffected.

</details>

## Setting an Intermediate Value

You can also modify intermediate values. Assigning to an operation's `.output` replaces its value for the rest of the forward. Here we zero out the MLP's post-GELU activations at layer 11 to see how it affects the prediction:

In [5]:
with model.trace("The Eiffel Tower is in the city of"):
    normal_logits = model.lm_head.output.save()

with model.trace("The Eiffel Tower is in the city of"):
    # Zero the MLP's GELU output at layer 11
    model.transformer.h[11].mlp.source.self_act_0.output[:] = 0
    modified_logits = model.lm_head.output.save()

print(f"Normal:      {model.tokenizer.decode(normal_logits[0, -1].argmax(dim=-1))}")
print(f"Zeroed GELU: {model.tokenizer.decode(modified_logits[0, -1].argmax(dim=-1))}")

Normal:       Paris
Zeroed GELU:  London


## Patching Between Layers

Transfer an intermediate value from one layer's operation to another:

In [6]:
with model.trace("The Eiffel Tower is in the city of"):
    # Capture layer 0's post-GELU activations
    gelu_0 = model.transformer.h[0].mlp.source.self_act_0.output

    # Patch them into layer 5
    model.transformer.h[5].mlp.source.self_act_0.output = gelu_0

    logits = model.lm_head.output.save()

print(f"Patched MLP prediction: {model.tokenizer.decode(logits[0, -1].argmax(dim=-1))}")

Patched MLP prediction:  London


## Recursive Source Tracing

`.source` works recursively. If a labeled operation calls a plain Python function, you can chain `.source` again to expose *its* operations. For example, GPT-2's attention module calls an `attention_interface` function, which internally calls `scaled_dot_product_attention` — here we drill in and read the query tensor it receives:

In [7]:
with model.trace("The Eiffel Tower is in the city of"):
    # Drill into the attention interface → the SDPA call's input (the query tensor)
    sdpa = model.transformer.h[0].attn.source.attention_interface_0.source
    query = sdpa.torch_nn_functional_scaled_dot_product_attention_0.input.save()

print(f"Query shape: {query.shape}")  # [batch, heads, seq_len, head_dim]

Query shape: torch.Size([1, 12, 10, 64])


Recursive `.source` is **trace-only** — the called function is resolved from the live value flowing through the call at run time, so `op.source` outside a trace raises `SourceNotAvailable`.

<details class="admonition tip">
<summary>Viewing a specific operation</summary>

Print a specific operation to see it highlighted in its surrounding context:

```python
print(model.transformer.h[0].mlp.source.self_c_proj_0)
```

This shows the operation flagged with `-->` / `<--` and surrounding lines for context.

</details>

<details class="admonition warning">
<summary>Don't chain .source into a submodule call</summary>

If a `.source` listing shows a submodule call (like `self.c_proj`), access that submodule directly — drilling into it with `.source` raises `SourceNotAvailable`:

```python
# Wrong — don't chain .source into a submodule call
model.transformer.h[0].mlp.source.self_c_proj_0.source

# Correct — access the submodule directly
model.transformer.h[0].mlp.c_proj.source
```

</details>